# RH-DINO evaluation

This notebook runs the evaluation of the RadHarmony paper for its three models, then shows the results next
to the numbers of the paper (Table 4).

- **RH-DINO:** the RH-DINO weights with the percentile scaling of its pretraining (input 512).
- **RAD-DINO:** the public checkpoint `microsoft/rad-dino` with its own image processor (input 518).
- **Original DINOv3:** `facebook/dinov3-vitb16-pretrain-lvd1689m` with the RadHarmony DINOv3 recipe (input 512).

- **VinDr-CXR:** a linear probe on the CLS token (1,500 training images, 5 seeds, 7 findings). The probe C is
  chosen on a validation split (3,000 fixed training images), then the model is tested on the official
  3,000-image test split.
- **SIIM-ACR:** a 1 x 1 conv head on the patch features, 5-fold cross-validation grouped by ChestX-ray14 patient.
- **Statistics:** 95% confidence intervals from a bootstrap over patients, and a paired bootstrap of the two
  models (`compare.py`).

Run it from `model/rh_dino/` on a machine with a GPU. Each step calls `evaluate.py` and writes to
`OUT/<task>/<name>/`. A step is skipped when its results are already there, so you can run the notebook again.
On one GPU, VinDr-CXR takes about 30 minutes per model and SIIM-ACR about 1.5 hours per model, with the image
cache. The cache needs about 30 GB (VinDr-CXR) and 40 GB (SIIM-ACR) per model.

## Settings

Download VinDr-CXR (PhysioNet) and SIIM-ACR Pneumothorax (Kaggle) yourself.

- `VINDR_DIR`: the VinDr-CXR release folder (`vindr-cxr/1.0.0`, with `train/`, `test/` and `annotations/`).
- `SIIM_CSV`: `train-rle.csv`. `SIIM_DICOM_DIR`: `dicom-images-train/`.
- `OUT`: the result folder. `CACHE`: the image cache.
- `DINOV3_512_DIR`: a new folder for the original DINOv3 at 512 pixels. The notebook makes it with
  `make_dinov3_512.py`. The Hugging Face model is gated: accept its license and log in (`hf auth login`) first.

In [1]:
RH_DINO_DIR = "/path/to/RH-DINO"
DINOV3_512_DIR = "/path/to/dinov3_512"
VINDR_DIR = "/path/to/vindr-cxr/1.0.0"
SIIM_CSV = "/path/to/SIIM-ACR/train-rle.csv"
SIIM_DICOM_DIR = "/path/to/SIIM-ACR/dicom-images-train"
OUT = "/path/to/rh_dino_eval"
CACHE = "/path/to/rh_dino_cache"
GPU = "0"
C_GRID = ["0.01", "0.1", "1", "10", "100", "1000"]

# name -> evaluate.py arguments of the model
MODELS = {
    "rh_dino": ["--backbone", "dinov3", "--hub", RH_DINO_DIR, "--scaling", "percentile"],
    "raddino": ["--backbone", "raddino"],
    "dinov3_original_512": ["--backbone", "dinov3", "--hub", DINOV3_512_DIR],
}

In [2]:
import os
import subprocess
import sys
from pathlib import Path

import pandas as pd

import compare

ENV = dict(os.environ, CUDA_VISIBLE_DEVICES=GPU)


def run(task, name, extra, done_file):
    """Run one evaluate.py step, unless OUT/<task>/<name>/<done_file> exists."""
    out = Path(OUT, task, name)
    if (out / done_file).exists():
        print(f"{task}/{name}: done before, skipped")
        return
    cmd = [sys.executable, "evaluate.py", "--task", task, "--name", name, "--out", OUT, "--cache", CACHE,
           *MODELS[name], *extra]
    print(" ".join(cmd))
    subprocess.run(cmd, check=True, env=ENV)


if not Path(DINOV3_512_DIR, "model.safetensors").exists():
    subprocess.run([sys.executable, "make_dinov3_512.py", DINOV3_512_DIR], check=True)

## 1. Choose the probe C (VinDr-CXR validation split)

Each model gets its own C, the best macro AUPRC of the grid. The test set does not choose C. In the paper, the
grid chose C = 10 for RH-DINO and RAD-DINO and C = 100 for the original DINOv3.

In [3]:
for name in MODELS:
    run("vindr_val", name, ["--vindr-dir", VINDR_DIR, "--C", *C_GRID], "best_C.json")
    print(name, "best C:", Path(OUT, "vindr_val", name, "best_C.json").read_text())

python evaluate.py --task vindr_val --name rh_dino --out /path/to/rh_dino_eval --cache /path/to/rh_dino_cache --backbone dinov3 --hub /path/to/RH-DINO --scaling percentile --vindr-dir /path/to/vindr-cxr/1.0.0 --C 0.01 0.1 1 10 100 1000


      C  auprc_mean  auroc_mean
   0.01    0.543941    0.927325
   0.10    0.572944    0.937989
   1.00    0.619621    0.947467
  10.00    0.634937    0.944234
 100.00    0.607933    0.931674
1000.00    0.589331    0.924955 
best C = 10
                label  n_train  auroc_mean  auroc_n  auroc_std  auroc_ci_lo  ...  mcc_ci_hi  threshold_mean  threshold_n  threshold_std  threshold_ci_lo  threshold_ci_hi
0  aortic_enlargement     1500    0.971016        5   0.003657     0.963225  ...        NaN             NaN            0            NaN              NaN              NaN
1        cardiomegaly     1500    0.971698        5   0.004168     0.962958  ...        NaN             NaN            0            NaN              NaN              NaN
2        lung_opacity     1500    0.912563        5   0.019425     0.874658  ...        NaN             NaN            0            NaN              NaN              NaN
3       macro_average     1500    0.944234        5   0.004158     0.936228  ...   

rh_dino best C: {"C": 10.0}
python evaluate.py --task vindr_val --name raddino --out /path/to/rh_dino_eval --cache /path/to/rh_dino_cache --backbone raddino --vindr-dir /path/to/vindr-cxr/1.0.0 --C 0.01 0.1 1 10 100 1000


      C  auprc_mean  auroc_mean
   0.01    0.550824    0.929806
   0.10    0.579299    0.938639
   1.00    0.613805    0.945349
  10.00    0.616685    0.939925
 100.00    0.586810    0.926252
1000.00    0.572538    0.920936 
best C = 10
                label  n_train  auroc_mean  auroc_n  auroc_std  auroc_ci_lo  ...  mcc_ci_hi  threshold_mean  threshold_n  threshold_std  threshold_ci_lo  threshold_ci_hi
0  aortic_enlargement     1500    0.956232        5   0.004824     0.946196  ...        NaN             NaN            0            NaN              NaN              NaN
1        cardiomegaly     1500    0.952299        5   0.006789     0.939397  ...        NaN             NaN            0            NaN              NaN              NaN
2        lung_opacity     1500    0.932501        5   0.014160     0.901538  ...        NaN             NaN            0            NaN              NaN              NaN
3       macro_average     1500    0.939925        5   0.003883     0.932252  ...   

raddino best C: {"C": 10.0}
python evaluate.py --task vindr_val --name dinov3_original_512 --out /path/to/rh_dino_eval --cache /path/to/rh_dino_cache --backbone dinov3 --hub /path/to/dinov3_512 --vindr-dir /path/to/vindr-cxr/1.0.0 --C 0.01 0.1 1 10 100 1000


      C  auprc_mean  auroc_mean
   0.01    0.310907    0.830217
   0.10    0.318006    0.834121
   1.00    0.375742    0.864738
  10.00    0.437555    0.890873
 100.00    0.460659    0.894245
1000.00    0.449585    0.883499 
best C = 100
                label  n_train  auroc_mean  auroc_n  auroc_std  auroc_ci_lo  ...  mcc_ci_hi  threshold_mean  threshold_n  threshold_std  threshold_ci_lo  threshold_ci_hi
0  aortic_enlargement     1500    0.935669        5   0.005686     0.924449  ...        NaN             NaN            0            NaN              NaN              NaN
1        cardiomegaly     1500    0.938854        5   0.006309     0.925855  ...        NaN             NaN            0            NaN              NaN              NaN
2        lung_opacity     1500    0.880719        5   0.021558     0.834645  ...        NaN             NaN            0            NaN              NaN              NaN
3       macro_average     1500    0.894245        5   0.007159     0.879969  ...  

dinov3_original_512 best C: {"C": 100.0}


## 2. VinDr-CXR test split

In [4]:
for name in MODELS:
    run("vindr", name, ["--vindr-dir", VINDR_DIR, "--C-from", str(Path(OUT, "vindr_val", name))],
        "results_summary.csv")

python evaluate.py --task vindr --name rh_dino --out /path/to/rh_dino_eval --cache /path/to/rh_dino_cache --backbone dinov3 --hub /path/to/RH-DINO --scaling percentile --vindr-dir /path/to/vindr-cxr/1.0.0 --C-from /path/to/rh_dino_eval/vindr_val/rh_dino


                label  n_train  auroc_mean  auroc_n  auroc_std  auroc_ci_lo  ...  mcc_ci_hi  threshold_mean  threshold_n  threshold_std  threshold_ci_lo  threshold_ci_hi
0  aortic_enlargement     1500    0.902687        5   0.011663     0.877619  ...        NaN             NaN            0            NaN              NaN              NaN
1        cardiomegaly     1500    0.947557        5   0.011803     0.923662  ...        NaN             NaN            0            NaN              NaN              NaN
2        lung_opacity     1500    0.818704        5   0.027029     0.762630  ...        NaN             NaN            0            NaN              NaN              NaN
3       macro_average     1500    0.892449        5   0.006725     0.879058  ...        NaN             NaN            0            NaN              NaN              NaN
4    pleural_effusion     1500    0.934375        5   0.017452     0.892039  ...        NaN             NaN            0            NaN              N

python evaluate.py --task vindr --name raddino --out /path/to/rh_dino_eval --cache /path/to/rh_dino_cache --backbone raddino --vindr-dir /path/to/vindr-cxr/1.0.0 --C-from /path/to/rh_dino_eval/vindr_val/raddino


                label  n_train  auroc_mean  auroc_n  auroc_std  auroc_ci_lo  ...  mcc_ci_hi  threshold_mean  threshold_n  threshold_std  threshold_ci_lo  threshold_ci_hi
0  aortic_enlargement     1500    0.910997        5   0.009636     0.893200  ...        NaN             NaN            0            NaN              NaN              NaN
1        cardiomegaly     1500    0.941885        5   0.008446     0.925513  ...        NaN             NaN            0            NaN              NaN              NaN
2        lung_opacity     1500    0.839749        5   0.029653     0.780867  ...        NaN             NaN            0            NaN              NaN              NaN
3       macro_average     1500    0.903339        5   0.006103     0.891620  ...        NaN             NaN            0            NaN              NaN              NaN
4    pleural_effusion     1500    0.960471        5   0.009544     0.938022  ...        NaN             NaN            0            NaN              N

python evaluate.py --task vindr --name dinov3_original_512 --out /path/to/rh_dino_eval --cache /path/to/rh_dino_cache --backbone dinov3 --hub /path/to/dinov3_512 --vindr-dir /path/to/vindr-cxr/1.0.0 --C-from /path/to/rh_dino_eval/vindr_val/dinov3_original_512


                label  n_train  auroc_mean  auroc_n  auroc_std  auroc_ci_lo  ...  mcc_ci_hi  threshold_mean  threshold_n  threshold_std  threshold_ci_lo  threshold_ci_hi
0  aortic_enlargement     1500    0.859962        5   0.013430     0.830481  ...        NaN             NaN            0            NaN              NaN              NaN
1        cardiomegaly     1500    0.897250        5   0.010021     0.876943  ...        NaN             NaN            0            NaN              NaN              NaN
2        lung_opacity     1500    0.745052        5   0.032494     0.681122  ...        NaN             NaN            0            NaN              NaN              NaN
3       macro_average     1500    0.827343        5   0.008201     0.811521  ...        NaN             NaN            0            NaN              NaN              NaN
4    pleural_effusion     1500    0.919650        5   0.019105     0.876943  ...        NaN             NaN            0            NaN              N

## 3. SIIM-ACR segmentation

In [5]:
for name in MODELS:
    run("siim", name, ["--siim-csv", SIIM_CSV, "--siim-dicom-dir", SIIM_DICOM_DIR], "seg_summary.csv")

python evaluate.py --task siim --name rh_dino --out /path/to/rh_dino_eval --cache /path/to/rh_dino_cache --backbone dinov3 --hub /path/to/RH-DINO --scaling percentile --siim-csv /path/to/SIIM-ACR/train-rle.csv --siim-dicom-dir /path/to/SIIM-ACR/dicom-images-train


SIIM-ACR: 10,675 images, 5,691 ChestX-ray14 patients


     metric      mean     ci_lo     ci_hi  n_images
0  dice_pos  0.348519  0.324809  0.371815      2379
1  giou_pos -0.331286 -0.363525 -0.298549      2379
2  dice_all  0.218998  0.199247  0.239148      3786


python evaluate.py --task siim --name raddino --out /path/to/rh_dino_eval --cache /path/to/rh_dino_cache --backbone raddino --siim-csv /path/to/SIIM-ACR/train-rle.csv --siim-dicom-dir /path/to/SIIM-ACR/dicom-images-train


SIIM-ACR: 10,675 images, 5,691 ChestX-ray14 patients


     metric      mean     ci_lo     ci_hi  n_images
0  dice_pos  0.313183  0.291189  0.334440      2379
1  giou_pos -0.434813 -0.462795 -0.407308      2379
2  dice_all  0.165349  0.148145  0.182409      4506


python evaluate.py --task siim --name dinov3_original_512 --out /path/to/rh_dino_eval --cache /path/to/rh_dino_cache --backbone dinov3 --hub /path/to/dinov3_512 --siim-csv /path/to/SIIM-ACR/train-rle.csv --siim-dicom-dir /path/to/SIIM-ACR/dicom-images-train


SIIM-ACR: 10,675 images, 5,691 ChestX-ray14 patients


     metric      mean     ci_lo     ci_hi  n_images
0  dice_pos  0.224691  0.205407  0.245115      2379
1  giou_pos -0.514084 -0.538213 -0.491392      2379
2  dice_all  0.107922  0.095702  0.122132      4953


## 4. Results

The paper columns are the numbers of the RadHarmony paper (Table 4 and the RH-DINO paragraph of the results).
Small differences in the last digit can come from the GPU (the SIIM-ACR heads are trained on the GPU).

In [6]:
PAPER = {  # test AUPRC, test AUROC, SIIM-ACR Dice
    "rh_dino": (0.445, 0.892, 0.349),
    "raddino": (0.467, 0.903, 0.313),
    "dinov3_original_512": (0.318, 0.827, 0.225),
}

rows = []
for name in MODELS:
    v = pd.read_csv(Path(OUT, "vindr", name, "results_summary.csv")).set_index("label").loc["macro_average"]
    s = pd.read_csv(Path(OUT, "siim", name, "seg_summary.csv")).set_index("metric").loc["dice_pos"]
    rows.append({
        "model": name,
        "test AUPRC [95% CI]": f"{v.auprc_mean:.3f} [{v.auprc_ci_lo:.3f}, {v.auprc_ci_hi:.3f}]",
        "paper AUPRC": PAPER[name][0],
        "test AUROC": round(v.auroc_mean, 3),
        "paper AUROC": PAPER[name][1],
        "SIIM-ACR Dice [95% CI]": f"{s['mean']:.3f} [{s.ci_lo:.3f}, {s.ci_hi:.3f}]",
        "paper Dice": PAPER[name][2],
    })
pd.DataFrame(rows).set_index("model")

,test AUPRC [95% CI],paper AUPRC,test AUROC,paper AUROC,SIIM-ACR Dice [95% CI],paper Dice
model,,,,,,
rh_dino,"0.445 [0.408, 0.485]",0.445,0.892,0.892,"0.349 [0.325, 0.372]",0.349
raddino,"0.467 [0.441, 0.499]",0.467,0.903,0.903,"0.313 [0.291, 0.334]",0.313
dinov3_original_512,"0.318 [0.289, 0.360]",0.318,0.827,0.827,"0.225 [0.205, 0.245]",0.225


### AUPRC per finding (VinDr-CXR test split)

In [7]:
per = {name: pd.read_csv(Path(OUT, "vindr", name, "results_summary.csv")).set_index("label")["auprc_mean"]
       for name in MODELS}
pd.DataFrame(per).round(3)

,rh_dino,raddino,dinov3_original_512
label,,,
aortic_enlargement,0.415,0.404,0.323
cardiomegaly,0.714,0.690,0.523
lung_opacity,0.135,0.138,0.086
macro_average,0.445,0.467,0.318
pleural_effusion,0.508,0.641,0.512
pleural_thickening,0.304,0.275,0.246
pulmonary_fibrosis,0.455,0.518,0.266
tuberculosis,0.581,0.600,0.269


### Paired comparison: RH-DINO minus RAD-DINO

Paper: VinDr-CXR macro AUPRC -0.022 (95% CI -0.052 to 0.006, p = 0.14); SIIM-ACR Dice +0.035 (95% CI 0.026
to 0.046, p < 0.001).

In [8]:
vindr = compare.vindr_diffs(Path(OUT, "vindr", "raddino"), Path(OUT, "vindr", "rh_dino"), "auprc")
vindr.round(4)

,label,A,B,diff,seed_min,seed_max,ci_lo,ci_hi,p,n_boot
0,aortic_enlargement,0.4041,0.4147,0.0106,-0.0116,0.0373,-0.0419,0.0688,0.7578,5000
1,cardiomegaly,0.6901,0.7140,0.0239,0.0001,0.0412,-0.0259,0.0747,0.3663,5000
2,lung_opacity,0.1381,0.1347,-0.0034,-0.0249,0.0182,-0.0523,0.0450,0.8422,5000
3,macro_average,0.4667,0.4445,-0.0221,-0.0416,-0.0089,-0.0523,0.0055,0.1380,5000
4,pleural_effusion,0.6414,0.5077,-0.1337,-0.2582,-0.0758,-0.2920,-0.0239,0.0144,5000
5,pleural_thickening,0.2750,0.3041,0.0291,-0.0214,0.0674,-0.0454,0.0966,0.4831,5000
6,pulmonary_fibrosis,0.5184,0.4554,-0.0630,-0.0897,-0.0410,-0.1215,-0.0024,0.0404,5000
7,tuberculosis,0.5998,0.5813,-0.0186,-0.0496,0.0148,-0.0836,0.0436,0.6043,5000


In [9]:
siim = compare.siim_diffs(Path(OUT, "siim", "raddino"), Path(OUT, "siim", "rh_dino"), n_boot=5000)
siim.round(4)

,metric,A,B,diff,ci_lo,ci_hi,p,n_images,n_boot
0,dice_pos,0.3132,0.3485,0.0353,0.0256,0.0459,0.0004,2379,5000
1,giou_pos,-0.4348,-0.3313,0.1035,0.0904,0.1172,0.0004,2379,5000
